# Классификация типа рака по факторам риска

Мультиклассовая классификация (5 типов рака: Breast, Colon, Lung, Prostate, Skin) на табличных данных
Cancer Risk Factors Dataset (Kaggle, 2000 пациентов, 21 столбец).

Что сделано:
1. EDA: пропуски, дубликаты, распределения, корреляции.
2. Предобработка: удаление идентификатора и «готовых» индексов риска, кодирование целевой переменной, стратифицированное разбиение.
3. Четыре модели: логистическая регрессия, KNN, случайный лес, XGBoost. Сравнение по кросс-валидации и на тесте.
4. Подбор гиперпараметров (GridSearchCV) и честная оценка, дал ли он что-то.
5. Интерпретация: permutation importance и SHAP.

**Важно.** Датасет учебный, скорее всего синтетический (проверьте описание на странице датасета). Выводы о реальных причинах рака из него делать нельзя.

## 1. Импорты и настройки

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost не установлен: ячейки с XGBoost будут пропущены (pip install xgboost)")

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Загрузка данных

Скачайте `cancer-risk-factors.csv` со страницы датасета на Kaggle и положите рядом с ноутбуком
(сам файл в репозиторий не добавляется). На Kaggle ноутбук найдёт файл по второму пути.

In [ ]:
CANDIDATE_PATHS = [
    Path("cancer-risk-factors.csv"),
    Path("/kaggle/cancer-risk-factors.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден cancer-risk-factors.csv. Положите его рядом с ноутбуком.")

df = pd.read_csv(DATA_PATH)
print("Размер:", df.shape)
display(df.head())

## 3. EDA

In [ ]:
info = pd.DataFrame({
    "Пропуски": df.isnull().sum(),
    "Уникальных": df.nunique(),
    "Тип": df.dtypes,
})
display(info)
print("Дубликатов строк:", df.duplicated().sum())
display(df.describe().T.round(2))

### 3.1 Целевая переменная и пол

Обратите внимание на график ниже: рак простаты бывает только у мужчин, а рак груди почти только у женщин.
Это значит, что `Gender` сильно помогает **различать типы рака**, но это не «фактор риска» в медицинском смысле,
а особенность самих классов. Дальше это проверяется отдельным экспериментом.

In [ ]:
order = df["Cancer_Type"].value_counts().index
gender_label = df["Gender"].map({0: "female", 1: "male"})  # кодировка из описания датасета

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="Cancer_Type", data=df, order=order, ax=axes[0])
axes[0].set_title("Количество пациентов по типам рака")
sns.countplot(x="Cancer_Type", hue=gender_label, data=df, order=order, ax=axes[1])
axes[1].set_title("Тип рака и пол")
axes[1].legend(title="Пол")
plt.tight_layout()
plt.show()

display(pd.crosstab(df["Cancer_Type"], gender_label))

### 3.2 Распределения признаков

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), ["Age", "BMI", "Smoking", "Air_Pollution", "Obesity", "Family_History"]):
    sns.histplot(df[col], bins=20 if col in ("Age", "BMI") else 11, kde=col in ("Age", "BMI"), ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

print("Доля пациентов с Family_History = 1: {:.1%}".format(df["Family_History"].mean()))
print("Доля значений Smoking = 10: {:.1%}".format((df["Smoking"] == 10).mean()))

### 3.3 Корреляции

Матрица строится только по признакам. `Overall_Risk_Score` и `Risk_Level` исключены: это рассчитанные индексы,
и корреляция с ними не говорит о реальном влиянии факторов (вывод был бы круговым).

In [ ]:
FEATURE_VIEW = df.drop(columns=["Patient_ID", "Cancer_Type", "Risk_Level", "Overall_Risk_Score"])
corr = FEATURE_VIEW.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляции признаков")
plt.show()

# Проверка пар, которые по названию кажутся дублями
print("corr(Physical_Activity, Physical_Activity_Level) = {:.2f}".format(corr.loc["Physical_Activity", "Physical_Activity_Level"]))
print("corr(BMI, Obesity) = {:.2f}".format(corr.loc["BMI", "Obesity"]))

# Самые сильные пары
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(key=np.abs, ascending=False)
display(pairs.head(5).round(2))

## 4. Предобработка

- `Patient_ID` — идентификатор, не несёт информации.
- `Overall_Risk_Score`, `Risk_Level` — рассчитанные индексы риска, их нельзя использовать как признаки (утечка).
- `Cancer_Type` кодируется `LabelEncoder`.
- Разбиение стратифицированное (25% на тест). Масштабирование делается внутри `Pipeline`, поэтому `StandardScaler`
  учится только на обучающей части (и внутри каждой фолды кросс-валидации).

In [ ]:
DROP_COLS = ["Patient_ID", "Overall_Risk_Score", "Risk_Level"]
X = df.drop(columns=DROP_COLS + ["Cancer_Type"])

le = LabelEncoder()
y = le.fit_transform(df["Cancer_Type"])
print("Классы:", dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 5. Модели и кросс-валидация

Сначала сравниваем модели по 5-кратной стратифицированной кросс-валидации **на обучающей выборке**.
Так оценка не зависит от одного случайного разбиения. Тест используется только для финальной проверки.

In [ ]:
models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(eval_metric="mlogloss", random_state=RANDOM_STATE)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "f1": "f1_weighted", "roc_auc": "roc_auc_ovr_weighted"}

rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        "Model": name,
        "CV accuracy": res["test_accuracy"].mean(),
        "± std": res["test_accuracy"].std(),
        "CV F1": res["test_f1"].mean(),
        "CV ROC-AUC": res["test_roc_auc"].mean(),
    })
cv_table = pd.DataFrame(rows).set_index("Model").round(3)
display(cv_table)

## 6. Оценка на тестовой выборке

In [ ]:
fitted, test_rows = {}, []
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted"),
        "Recall": recall_score(y_test, pred, average="weighted"),
        "F1": f1_score(y_test, pred, average="weighted"),
        "ROC-AUC": roc_auc_score(y_test, proba, multi_class="ovr", average="weighted"),
    })
test_table = pd.DataFrame(test_rows).set_index("Model").round(3)
display(test_table)

# Насколько можно доверять разнице между моделями
n_test = len(y_test)
acc = test_table["Accuracy"].max()
se = np.sqrt(acc * (1 - acc) / n_test)
print(f"Размер теста: {n_test}. Стандартная ошибка accuracy ≈ ±{se:.3f}.")
print("Разницу между моделями меньше этой величины нельзя считать значимой.")

In [ ]:
names_to_plot = [n for n in ["RandomForest", "XGBoost"] if n in fitted]
fig, axes = plt.subplots(1, len(names_to_plot), figsize=(6 * len(names_to_plot), 5))
axes = np.atleast_1d(axes)
for ax, name in zip(axes, names_to_plot):
    ConfusionMatrixDisplay.from_predictions(
        y_test, fitted[name].predict(X_test), display_labels=le.classes_,
        normalize="true", values_format=".2f", cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(f"{name}: доля верных ответов по классам")
plt.tight_layout()
plt.show()

best_name = test_table["F1"].idxmax()
print(f"Лучшая модель по F1 на тесте: {best_name}\n")
print(classification_report(y_test, fitted[best_name].predict(X_test), target_names=le.classes_))

### 6.1 Проверка: что даёт признак Gender

Если пол в основном различает «мужские» и «женские» типы рака, то без него точность должна упасть заметно,
но не потому, что пол вызывает рак, а потому что он выдаёт класс.

In [ ]:
rf_with = fitted["RandomForest"]
rf_without = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE).fit(X_train.drop(columns="Gender"), y_train)

acc_with = accuracy_score(y_test, rf_with.predict(X_test))
acc_without = accuracy_score(y_test, rf_without.predict(X_test.drop(columns="Gender")))
print(f"RandomForest accuracy с Gender:    {acc_with:.3f}")
print(f"RandomForest accuracy без Gender:  {acc_without:.3f}")

# Recall по каждому классу: какие типы рака страдают сильнее всего
per_class = pd.DataFrame({
    "с Gender": recall_score(y_test, rf_with.predict(X_test), average=None),
    "без Gender": recall_score(y_test, rf_without.predict(X_test.drop(columns="Gender")), average=None),
}, index=le.classes_).round(2)
display(per_class)

## 7. Подбор гиперпараметров

Подбор идёт по кросс-валидации на обучающей выборке. Сравниваем и CV-метрику, и тест.
Если улучшение меньше стандартной ошибки (см. выше), считаем, что подбор ничего не дал.

In [ ]:
def tune(estimator, params, name):
    grid = GridSearchCV(estimator, params, cv=cv, scoring="f1_weighted", n_jobs=-1)
    grid.fit(X_train, y_train)
    f1_before = f1_score(y_test, fitted[name].predict(X_test), average="weighted")
    f1_after = f1_score(y_test, grid.best_estimator_.predict(X_test), average="weighted")
    print(f"{name}: лучшие параметры {grid.best_params_}")
    print(f"  CV F1: базовая {cv_table.loc[name, 'CV F1']:.3f} -> после подбора {grid.best_score_:.3f}")
    print(f"  Test F1: до {f1_before:.3f} -> после {f1_after:.3f}\n")
    return grid

rf_grid = tune(
    RandomForestClassifier(random_state=RANDOM_STATE),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20],
     "min_samples_split": [2, 5], "min_samples_leaf": [1, 2]},
    "RandomForest",
)

if HAS_XGB:
    xgb_grid = tune(
        XGBClassifier(eval_metric="mlogloss", random_state=RANDOM_STATE),
        {"n_estimators": [100, 200], "max_depth": [3, 5, 7],
         "learning_rate": [0.01, 0.1], "subsample": [0.8, 1.0]},
        "XGBoost",
    )

## 8. Важность признаков

Используем **permutation importance** на тестовой выборке: смотрим, насколько падает F1, если случайно перемешать значения одного признака.
Она надёжнее встроенных `feature_importances_`, которые завышают важность признаков с большим числом значений.

In [ ]:
final_name = "XGBoost" if HAS_XGB else "RandomForest"
final_model = xgb_grid.best_estimator_ if HAS_XGB else rf_grid.best_estimator_

perm = permutation_importance(final_model, X_test, y_test, scoring="f1_weighted",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_df = pd.DataFrame({"mean": perm.importances_mean, "std": perm.importances_std}, index=X.columns)
perm_df = perm_df.sort_values("mean")

plt.figure(figsize=(8, 6))
plt.barh(perm_df.index, perm_df["mean"], xerr=perm_df["std"])
plt.xlabel("Падение F1 при перемешивании признака")
plt.title(f"Permutation importance ({final_name})")
plt.tight_layout()
plt.show()

## 9. SHAP

Для мультиклассовой модели SHAP возвращает значения отдельно для каждого класса. В разных версиях `shap`
это список массивов или один трёхмерный массив `(объекты, признаки, классы)`. Ниже оба случая приводятся к списку
по классам, иначе `summary_plot` ошибочно строит график взаимодействий вместо обычного.

In [ ]:
try:
    import shap
except ImportError:
    shap = None
    print("shap не установлен (pip install shap): ячейка пропущена")

if HAS_XGB and shap is not None:
    explainer = shap.TreeExplainer(final_model)
    sv = explainer.shap_values(X_test)
    if isinstance(sv, list):
        sv_by_class = sv
    else:
        sv = np.asarray(sv)
        sv_by_class = [sv[:, :, k] for k in range(sv.shape[2])]

    # Общая важность: среднее |SHAP| по всем классам
    shap.summary_plot(sv_by_class, X_test, plot_type="bar", class_names=list(le.classes_))

    # Детальный график для одного класса
    class_to_show = "Lung"
    k = list(le.classes_).index(class_to_show)
    print(f"SHAP для класса {class_to_show}: красный = высокое значение признака")
    shap.summary_plot(sv_by_class[k], X_test)

## 10. Выводы и ограничения

**Результаты** (числа см. в таблицах выше):
- Деревья (случайный лес, XGBoost) работают лучше линейной модели и заметно лучше KNN.
- Различия между случайным лесом и XGBoost меньше стандартной ошибки на тесте, поэтому ставить одну выше другой нельзя.
- Подбор гиперпараметров не дал значимого улучшения: базовые настройки уже близки к потолку для этих данных.
- Хуже всего распознаётся рак кожи: модель путает его с раком груди и простаты.

**Про признаки:**
- `Gender` сильно влияет на предсказание из-за структуры классов (простата — мужчины, грудь — почти только женщины). Это не медицинский вывод.
- Корреляции с `Overall_Risk_Score` ничего не говорят о реальных факторах риска: индекс рассчитан по самим признакам.

**Ограничения:**
- Данные, вероятно, синтетические; оценка по одному разбиению и 500 тестовым пациентам.
- Признаки — готовые шкалы 0–10, а не измерения, поэтому интерпретация условна.
- Задача «предсказать тип рака по факторам риска» учебная и не имеет клинического применения.

**Что можно сделать дальше:** калибровка вероятностей, сравнение с градиентным бустингом из LightGBM/CatBoost, анализ ошибок
по классу Skin, повторить на реальных данных.